In [ ]:
# matplotlib 시각화를 주피터 노트북 내에서 직접 볼 수 있도록 설정하는 매직 명령어
%matplotlib inline


분류기(Classifier) 학습하기
===========================

지금까지 어떻게 신경망을 정의하고, 손실을 계산하며 또 가중치를 갱신하는지에
대해서 배웠습니다.

이제 아마도 이런 생각을 하고 계실텐데요,

데이터는 어떻게 하나요?
------------------------

일반적으로 이미지나 텍스트, 오디오나 비디오 데이터를 다룰텐데요, 이러한 데이터는
표준 Python 패키지를 사용하여 불러온 후 NumPy 배열로 변환하면 됩니다.
그리고 그 배열을 ``torch.*Tensor`` 로 변환하면 됩니다.

-  이미지는 Pillow나 OpenCV 같은 패키지가 유용합니다.
-  오디오를 처리할 때는 SciPy와 LibROSA가 유용하고요.
-  텍스트의 경우에는 그냥 Python이나 Cython의 것들을 사용하거나, NLTK나 SpaCy도
   좋습니다.

특별히 영상 분야를 위해서는 ``torchvision`` 이라는 패키지를 만들어두었는데요,
여기에는 Imagenet이나 CIFAR10, MNIST 등과 같은 일반적으로 사용하는 데이터셋을
불러오는 함수들(data loaders)이나, image, viz., ``torchvision.datasets`` 와
``torch.utils.data.DataLoader`` 데이터 변환기가 포함되어 있습니다.

이러한 기능은 엄청나게 편리하며, 매번 유사한 코드(boilerplate code)를 반복해서
작성하는 것을 피할 수 있습니다.

이 튜토리얼에서는 CIFAR10 데이터셋을 사용할 텐데요, 여기에는 다음과 같은 분류들이
있습니다: '비행기(airplane)', '자동차(automobile)', '새(bird)', '고양이(cat)',
'사슴(deer)', '개(dog)', '개구리(frog)', '말(horse)', '배(ship)', '트럭(truck)'.
그리고 CIFAR10에 포함된 이미지의 크기는 3x32x32인데요, 이는 32x32 픽셀 크기의 이미지가
3개 채널(channel)로 이뤄져 있다는 뜻입니다.
- 60000장의 컬러이미지로 구성.
- 50000장은 학습용, 10000장은 테스트용
  

![](https://tutorials.pytorch.kr/_images/cifar10.png)


## 이미지 분류기 학습하기

다음의 단계로 진행해보겠습니다:

1. `torchvision` 을 사용하여 CIFAR10의 학습용 / 시험용 데이터셋을 불러오고, 정규화(nomarlizing)합니다.
2. 합성곱 신경망(Convolution Neural Network)을 정의합니다.
3. 손실 함수를 정의합니다.
4. 학습용 데이터를 사용하여 신경망을 학습합니다.
5. 시험용 데이터를 사용하여 신경망을 검사합니다.



### 1. CIFAR10를 불러오고 정규화하기

``torchvision`` 을 사용하면 매우 쉽게 CIFAR10 데이터를 불러올 수 있습니다.

In [ ]:
# 딥러닝 프레임워크인 PyTorch와 이미지 처리를 위한 torchvision 라이브러리를 임포트합니다.
import torch
import torchvision
# torchvision 내의 transforms는 이미지 전처리(augmentation, normalization 등) 기능을 제공합니다.
import torchvision.transforms as transforms

torchvision 데이터셋의 출력(output)은 [0, 1] 범위를 갖는 PILImage 이미지입니다.
이를 [-1, 1]의 범위로 정규화된 Tensor로 변환하겠습니다.

> [참고]
> 만약 Windows 환경에서 BrokenPipeError가 발생한다면, torch.utils.data.DataLoader()의 num_worker를 0으로 설정해보세요.


In [ ]:
# 1. 이미지 전처리 규칙을 정의합니다. transforms.Compose를 사용해 여러 단계를 순차적으로 실행합니다.
transform = transforms.Compose(
    [# 2. PIL Image나 NumPy 배열 형태의 이미지를 PyTorch가 다룰 수 있는 Tensor로 변환합니다.
     #    이 과정에서 픽셀 값의 범위를 [0, 255]에서 [0.0, 1.0]으로 자동으로 조정(스케일링)합니다.
     transforms.ToTensor(),

     # 3. 텐서의 값을 정규화(Normalize)합니다.
     #    (R, G, B) 각 채널에 대해 평균(0.5)을 빼고 표준편차(0.5)로 나누어, 픽셀 값의 범위를 [-1.0, 1.0]으로 조정합니다.
     #    이는 모델의 학습을 더 안정적으로 만들어줍니다.
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# CIFAR-10 학습용 데이터셋을 다운로드하고, 위에서 정의한 전처리(transform)를 적용합니다.
# root: 데이터가 저장될 경로
# train=True: 학습용 데이터셋임을 명시
# download=True: 해당 경로에 데이터가 없으면 자동으로 다운로드
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

# DataLoader는 데이터셋을 모델에 효율적으로 공급하는 역할을 합니다.
# trainset: 사용할 데이터셋
# batch_size=4: 한 번에 4개의 이미지를 묶어서 모델에 전달
# shuffle=True: 각 에폭(epoch)마다 데이터의 순서를 무작위로 섞어 모델이 데이터 순서에 과적합되는 것을 방지
# num_workers=2: 데이터 로딩에 사용할 CPU 프로세서의 수
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

# CIFAR-10 테스트용 데이터셋을 동일한 방식으로 준비합니다.
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
# 테스트 시에는 데이터 순서를 섞을 필요가 없으므로 shuffle=False로 설정합니다.
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=2)

# 분류할 10개의 클래스 이름을 튜플로 정의합니다.
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

재미삼아 학습용 이미지 몇 개를 보겠습니다.



In [ ]:
# 2. 데이터 시각화
import matplotlib.pyplot as plt
import numpy as np

# 이미지 시각화를 위한 함수 정의
def imshow(img):
    # 정규화된 이미지 픽셀 값(-1.0 ~ 1.0)을 다시 원래 범위(0.0 ~ 1.0)로 되돌립니다. (unnormalize)
    img = img / 2 + 0.5
    # PyTorch 텐서를 시각화를 위해 NumPy 배열로 변환합니다.
    npimg = img.numpy()
    # PyTorch의 텐서 차원 순서(채널, 높이, 너비)를 Matplotlib이 인식하는 순서(높이, 너비, 채널)로 변경합니다.
    plt.imshow(np.transpose(npimg, (1, 2, 0)))


# 학습용 데이터 로더에서 이터레이터(iterator)를 생성합니다.
dataiter = iter(trainloader)
# 이터레이터에서 이미지와 레이블 한 배치를 가져옵니다.
images, labels = next(dataiter)

# torchvision.utils.make_grid 함수는 여러 이미지를 하나의 그리드 이미지로 만들어줍니다.
imshow(torchvision.utils.make_grid(images))

# 가져온 배치에 포함된 이미지들의 실제 정답(label)을 출력합니다.
batch_size = 4
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(batch_size)))

### 2. 합성곱 신경망(Convolution Neural Network) 정의하기

이전에 배웠던 신경망 섹션에서 신경망을 복사하고, (기존에 1채널 이미지만 처리하던
것 대신) 3채널 이미지를 처리할 수 있도록 수정합니다.



In [ ]:
# 3. CNN 모델 정의

import torch.nn as nn
import torch.nn.functional as F

# nn.Module을 상속받아 CNN 모델 클래스를 정의합니다.
class Net(nn.Module):
    # 모델의 구조와 필요한 레이어를 정의하는 __init__ 메서드
    def __init__(self):
        super(Net, self).__init__()
        # 첫 번째 합성곱 레이어 (nn.Conv2d)
        # in_channels=3: 입력 이미지의 채널 수 (RGB)
        # out_channels=6: 사용할 필터의 개수, 즉 출력 특징 맵(feature map)의 개수
        # kernel_size=5: 5x5 크기의 필터(커널)
        self.conv1 = nn.Conv2d(3, 6, 5)

        # 최대 풀링 레이어 (nn.MaxPool2d)
        # kernel_size=2, stride=2: 2x2 크기의 윈도우로 특징 맵을 훑으며, 2칸씩 이동합니다.
        # 이 과정을 통해 특징 맵의 가로, 세로 크기가 절반으로 줄어듭니다.
        self.pool = nn.MaxPool2d(2, 2)
        # 두 번째 합성곱 레이어
        # in_channels=6: 이전 conv1 레이어의 출력 채널 수와 동일해야 합니다.
        # out_channels=16: 16개의 필터를 사용해 16개의 특징 맵을 생성합니다.
        self.conv2 = nn.Conv2d(6, 16, 5)

        # 완전 연결 레이어 (nn.Linear, MLP) - 분류기 역할
        # in_features=16 * 5 * 5: 입력으로 들어오는 1차원 벡터의 크기.
        # (32x32 -> conv1(28x28) -> pool1(14x14) -> conv2(10x10) -> pool2(5x5))
        # 최종 특징 맵은 16개 채널의 5x5 크기이므로, 총 16*5*5=400개의 노드가 됩니다.
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        # 최종 출력은 10개 클래스에 대한 점수(score)여야 하므로 out_features=10
        self.fc3 = nn.Linear(84, 10)

    # 데이터가 모델을 통과하는 순서, 즉 순전파(forward pass) 과정을 정의하는 메서드
    def forward(self, x):
        # Conv1 -> ReLU -> Pool
        x = self.pool(F.relu(self.conv1(x)))
        # Conv2 -> ReLU -> Pool
        x = self.pool(F.relu(self.conv2(x)))
        # 2차원 특징 맵을 1차원 벡터로 평탄화(flatten)합니다.
        # start_dim=1: 배치 차원(0)은 유지하고 나머지 차원(1, 2, 3)을 하나로 합칩니다.
        x = torch.flatten(x, 1)
        # 완전 연결 레이어 통과
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 정의한 Net 클래스를 이용해 모델 객체(net)를 생성합니다.
net = Net()

### 3. 손실 함수와 Optimizer 정의하기

이제, 분류에 대한 교차 엔트로피 손실(Cross-Entropy loss)과 momentum을 갖는 SGD를 사용합니다.



In [ ]:
# 4. 손실 함수와 옵티마이저 정의

import torch.optim as optim

# 손실 함수(Loss Function)로 다중 클래스 분류에 널리 사용되는 교차 엔트로피 손실을 사용합니다.
criterion = nn.CrossEntropyLoss()
# 옵티마이저(Optimizer)로 경사 하강법(SGD)을 사용합니다.
# net.parameters(): 모델의 모든 학습 가능한 파라미터(가중치, 편향)를 최적화 대상으로 지정
# lr=0.001: 학습률(learning rate)
# momentum=0.9: 이전 업데이트의 방향을 일정 비율 유지하여, 더 빠르고 안정적인 수렴을 돕는 기법
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

### 4. 신경망 학습하기

이제부터 흥미로우실 겁니다.
데이터를 반복해서 신경망에 입력으로 제공하고, 최적화(Optimize)만 하면 됩니다.



In [ ]:
# 5. 신경망 학습 (Training Loop)
# 전체 데이터셋을 2번 반복하여 학습합니다 (epoch=2).
for epoch in range(2):

    # 손실 값을 누적하여 일정 주기마다 평균을 계산하기 위한 변수
    running_loss = 0.0
    # trainloader에서 미니배치 단위로 데이터를 가져옵니다. i는 배치의 인덱스입니다.
    for i, data in enumerate(trainloader, 0):
        # data는 [입력 이미지 텐서, 정답 레이블 텐서]로 구성된 리스트입니다.
        inputs, labels = data

        # 1. 기울기 초기화: PyTorch는 기울기를 누적하므로, 새로운 배치를 학습하기 전에 항상 0으로 초기화해야 합니다.
        optimizer.zero_grad()

        # 2. 순전파(Forward Pass): 모델에 입력을 넣어 예측값을 계산합니다.
        outputs = net(inputs)
        # 3. 손실 계산: 예측값과 실제 정답 레이블 간의 오차(손실)를 계산합니다.
        loss = criterion(outputs, labels)
        # 4. 역전파(Backward Pass): 손실을 바탕으로 각 파라미터에 대한 기울기를 계산합니다.
        loss.backward()
        # 5. 가중치 업데이트: 계산된 기울기를 이용해 옵티마이저가 모델의 가중치를 업데이트합니다.
        optimizer.step()

        # 학습 과정 통계 출력
        # loss.item()은 텐서에서 스칼라 값만 추출합니다.
        running_loss += loss.item()
        # 2000개의 미니배치를 처리할 때마다 평균 손실을 출력합니다.
        if i % 2000 == 1999:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            # running_loss를 다시 0으로 초기화합니다.
            running_loss = 0.0

print('Finished Training')

### 5. 시험용 데이터로 신경망 검사하기

학습용 데이터셋을 2회 반복하여 신경망을 학습시켰는데요, 신경망이 전혀 배운게
없을지도 모르니 확인해보겠습니다.

신경망이 예측한 정답과 진짜 정답(Ground-truth)을 비교하는 방식으로 확인할텐데요,
예측이 맞다면 샘플을 '맞은 예측값(Correct predictions)'에 넣겠습니다.

첫번째로 시험용 데이터를 좀 보겠습니다.



In [ ]:
# 6. 모델 평가

# 테스트 데이터 로더에서 이터레이터를 생성합니다.
dataiter = iter(testloader)
# 테스트용 이미지와 레이블 한 배치를 가져옵니다.
images, labels = next(dataiter)

# 가져온 테스트 이미지들을 시각화합니다.
imshow(torchvision.utils.make_grid(images))
# 해당 이미지들의 실제 정답(GroundTruth)을 출력합니다.
print('GroundTruth: ', ' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))

좋습니다, 이제 신경망이 어떻게 예측했는지를 보죠:



In [ ]:
# 학습된 모델에 테스트 이미지를 입력하여 예측을 수행합니다.
outputs = net(images)

출력은 10개 분류 각각에 대한 값으로 나타납니다. 어떤 분류에 대해서 더 높은 값이
나타난다는 것은, 신경망이 그 이미지가 더 해당 분류에 가깝다고 생각한다는 것입니다.
따라서, 가장 높은 값을 갖는 인덱스(index)를 뽑아보겠습니다:



In [ ]:
# torch.max 함수는 (최대값, 최대값의 인덱스)를 반환합니다.
# 우리는 클래스 예측이므로 인덱스만 필요합니다. '_'는 값을 무시하겠다는 의미입니다.
# dim=1: 각 행(이미지)에 대해 가장 큰 값을 가진 열(클래스)의 인덱스를 찾습니다.
_, predicted = torch.max(outputs.data, 1)

# 모델이 예측한 클래스를 출력합니다.
print('Predicted: ', ' '.join(f'{classes[predicted[j]]:5s}'
                              for j in range(4)))

결과가 괜찮아보이네요.

그럼 전체 데이터셋에 대해서는 어떻게 동작하는지 보겠습니다.



In [ ]:
# 전체 테스트 데이터셋에 대한 정확도를 계산합니다.
correct = 0
total = 0
# with torch.no_grad() 블록 내에서는 기울기를 계산하지 않아, 불필요한 연산을 줄이고 메모리를 절약합니다.
# 모델 평가 시에는 반드시 사용해야 합니다.
with torch.no_grad():
    for data in testloader:
        images, labels = data
        # 신경망에 이미지를 통과시켜 출력을 계산합니다.
        outputs = net(images)
        # 가장 높은 점수를 가진 클래스를 예측값으로 선택합니다.
        _, predicted = torch.max(outputs.data, 1)
        # 전체 레이블 수(배치 크기)를 누적합니다.
        total += labels.size(0)
        # 예측이 맞은 개수를 누적합니다. (predicted == labels)는 boolean 텐서를 반환합니다.
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

(10가지 분류에서 무작위로) 찍었을 때의 정확도인 10% 보다는 나아보입니다.
신경망이 뭔가 배우긴 한 것 같네요.

그럼 어떤 것들을 더 잘 분류하고, 어떤 것들을 더 못했는지 알아보겠습니다:



In [ ]:
# 각 클래스별로 맞춘 개수와 전체 개수를 저장할 딕셔너리를 준비합니다.
correct_pred = {classname: 0 for classname in classes}
total_pred = {classname: 0 for classname in classes}

# 기울기를 계산하지 않도록 설정합니다.
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = net(images)
        _, predictions = torch.max(outputs, 1)
        # zip을 이용해 각 이미지의 실제 레이블과 예측 레이블을 하나씩 확인합니다.
        for label, prediction in zip(labels, predictions):
            # 만약 예측이 맞았다면, 해당 클래스의 correct_pred 값을 1 증가시킵니다.
            if label == prediction:
                correct_pred[classes[label]] += 1
            # 해당 클래스의 전체 개수(total_pred)를 1 증가시킵니다.
            total_pred[classes[label]] += 1


# 각 클래스별 정확도를 계산하고 출력합니다.
for classname, correct_count in correct_pred.items():
    accuracy = 100 * float(correct_count) / total_pred[classname]
    print(f'Accuracy for class: {classname:5s} is {accuracy:.1f} %')

## GPU에서 학습하기
자, 이제 다음은 뭘까요?

이러한 신경망들을 GPU에서 실행한다면 어떨까요?

GPU에서 학습하기
Tensor를 GPU로 옮겼던 것처럼, 신경망을 GPU로 옮길 수 있습니다.
먼저 (CUDA를 사용할 수 있다면) 첫번째 CUDA 장치를 사용하도록 설정합니다:

In [ ]:
# torch.cuda.is_available() 함수로 CUDA 지원 GPU의 사용 가능 여부를 확인합니다.
# 사용 가능하면 'cuda:0' (첫 번째 GPU)을, 아니면 'cpu'를 device로 설정합니다.
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 설정된 장치(device)를 출력합니다.
print(device)

이 섹션의 나머지 부분에서는 device 를 CUDA 장치라고 가정하겠습니다.

그리고 이 메소드(Method)들은 재귀적으로 모든 모듈의 매개변수와 버퍼를 CUDA tensor로 변경합니다:

In [ ]:
# .to(device) 메서드를 사용해 모델의 모든 파라미터와 버퍼를 지정된 장치(GPU 또는 CPU)로 이동시킵니다.
net.to(device)

또한, 각 단계에서 입력(input)과 정답(target)도 GPU로 보내야 한다는 것도 기억해야 합니다:

In [ ]:
# 모델뿐만 아니라, 학습에 사용될 모든 데이터 텐서(입력, 레이블)도 동일한 장치로 이동시켜야 합니다.
inputs, labels = data[0].to(device), data[1].to(device)

CPU와 비교했을 때 어마어마한 속도 차이가 나지 않는 것은 왜 그럴까요?
그 이유는 바로 신경망이 너무 작기 때문입니다.

**연습:** 신경망의 크기를 키웠을 때 얼마나 빨라지는지 확인해보세요.
(첫번째 ``nn.Conv2d`` 의 2번째 매개변수와 두번째 ``nn.Conv2d`` 의 1번째
매개변수는 같아야 합니다.)




1. 신경망 구조 수정하기 (모델 크기 키우기)
먼저, 우리 모델의 설계도인 Net 클래스 코드를 찾아 수정해야 합니다.

- 수정할 코드 위치: 주피터 노트북에서 `2. 합성곱 신경망(Convolution Neural Network) 정의하기` 섹션 아래에 있는 class `Net(nn.Module):` 코드 블록을 찾으세요.

- 수정 방법: `self.conv1`과 `self.conv2` 레이어의 채널 수를 늘려주면 됩니다. 채널 수를 늘리는 것은 필터의 개수를 늘리는 것과 같으며, 이는 모델이 더 다양한 특징을 학습하도록 만들어 모델의 크기를 키우는 효과를 줍니다.

    ```python
    # [기존 코드]
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(3, 6, 5)      # 3채널 입력, 6채널 출력
            self.pool = nn.MaxPool2d(2, 2)
            self.conv2 = nn.Conv2d(6, 16, 5)     # 6채널 입력, 16채널 출력
            self.fc1 = nn.Linear(16 * 5 * 5, 120)
            # ... (이하 동일)


    # [수정 예시] 예를 들어, 채널 수를 약 10배씩 늘려보겠습니다. 
    # conv1의 출력 채널(out_channels)과 conv2의 입력 채널(in_channels)은 항상 같아야 합니다.
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            # 채널 수를 6 -> 64로 늘립니다.
            self.conv1 = nn.Conv2d(3, 64, 5)
            self.pool = nn.MaxPool2d(2, 2)
            # conv1의 출력 채널 수에 맞춰 입력 채널을 64로, 출력 채널은 16 -> 128로 늘립니다.
            self.conv2 = nn.Conv2d(64, 128, 5)
            # fc1의 입력 특성 수도 변경된 채널 수에 맞춰 수정해야 합니다. (128 * 5 * 5)
            self.fc1 = nn.Linear(128 * 5 * 5, 120)
            self.fc2 = nn.Linear(120, 84)
            self.fc3 = nn.Linear(84, 10)
    # ... (forward 메서드는 동일)
    ```

---

**2. 코드 재실행하기**
모델 구조가 변경되었으므로, **모델을 새로 생성하고 학습시키는 모든 과정을 다시 실행**해야 합니다.

아래 순서대로 코드 셀을 실행(`Shift` + `Enter`)하세요.

1. **모델 정의 셀 실행**: 방금 수정한 `class Net(nn.Module):` 와 `net = Net()`가 포함된 셀을 실행하여 새로운 구조의 모델을 생성합니다.

2. **손실 함수와 옵티마이저 정의 셀 실행**: `criterion = nn.CrossEntropyLoss()` 와 `optimizer = optim.SGD(...)`가 포함된 셀을 다시 실행하여 새로운 모델의 파라미터를 옵티마이저에 등록합니다.

3. **(선택) GPU 설정 셀 실행**: 만약 GPU를 사용한다면 `device = torch.device(...)` 와 `net.to(device)` 셀을 실행하여 모델을 GPU로 보냅니다.

4. **신경망 학습 셀 실행**: `for epoch in range(2):` 로 시작하는 학습 루프 셀을 실행하여 더 커진 모델을 새로 학습시킵니다.

5. **신경망 검사 셀 실행**: 학습이 끝난 후, 테스트 데이터로 성능을 평가하는 모든 셀을 순서대로 실행합니다.

---

**3. 결과 확인 및 토의**
- **학습 속도 비교:** 모델이 커졌기 때문에 GPU를 사용했을 때와 CPU를 사용했을 때의 학습 시간 차이가 이전보다 훨씬 더 명확하게 나타나는 것을 확인합니다.

- **성능(정확도) 비교:** 모델의 크기가 커지면서 학습해야 할 파라미터가 늘어나, 동일한 학습 횟수(epoch)에도 불구하고 이전보다 더 높은 정확도를 보이는지 확인하고 토의해볼 수 있습니다. (물론, 항상 성능이 좋아지는 것은 아니라는 점도 알고 있어야 합니다.)


**다음 목표를 달성했습니다:**

높은 수준에서 PyTorch의 Tensor library와 신경망를 이해합니다.
이미지를 분류하는 작은 신경망을 학습시킵니다.

### 여러개의 GPU에서 학습하기
모든 GPU를 활용해서 더욱 더 속도를 올리고 싶다면, [선택 사항: 데이터 병렬 처리 (Data Parallelism)](https://tutorials.pytorch.kr/beginner/blitz/data_parallel_tutorial.html) 을 참고하세요.

### 이 무엇을 해볼까요?
> https://tutorials.pytorch.kr/beginner/blitz/cifar10_tutorial.html?highlight=cifar#id7
- 비디오 게임을 할 수 있는 신경망 학습시키기
- imagenet으로 최첨단(state-of-the-art) ResNet 신경망 학습시키기

- 적대적 생성 신경망으로 얼굴 생성기 학습시키기

- 순환 LSTM 네트워크를 사용해 단어 단위 언어 모델 학습시키기

- 다른 예제들 참고하기

- 더 많은 튜토리얼 보기

- 포럼에서 PyTorch에 대해 얘기하기

- Slack에서 다른 사용자와 대화하기